# Sentence Transformer
이번에는 단어가 아닌, 문장 자체를 RoBERTa 기반으로 임베딩하여 유사한 문장을 찾고, 유사한 문서를 클러스터링 하는 실습입니다.

In [1]:
!pip install sentence-transformers datasets==2.21.0

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.2/255.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 2.3 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer, models

model_name = "klue/roberta-base"

# 단어 토큰을 임베딩 할 모델 가져오기.
embedding_model = models.Transformer(model_name)

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Pooler 정의
모델로 부터 추출된 토큰 단위 임베딩을 가지고 문장 임베딩을 어떻게 계산할 것인지를 결정하는 Pooler를 정의합니다. Pooling 기법은 여러 가지가 있습니다. 여기서는 Mean Pooling을 사용합니다.

Mean Pooling은 모델이 반환한 모든 토큰 임베딩에 대한 평균을 구하여 문장의 임베딩 벡터로 활용하는 기법입니다.

In [6]:
pooler = models.Pooling(
    embedding_model.get_word_embedding_dimension(), # 단어 임베딩 모델의 차원을 가져오기
    pooling_mode_mean_tokens=True, # 풀링을 평균으로 할 수 있게 설정함
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False
)

문장 내 토큰에 대한 임베딩을 수행할 Embedder와 Embedder에 의해 구해진 임베딩 벡터를 문장에 대한 임베딩 벡터로 만들어줄 Pooler를 정의 했으니, SentenceTransformer를 정의할 수 있습니다.

In [7]:
# modules에 정의되는 리스트 내의 원소 순서대로 임베딩 과정이 수행됩니다
#  즉 문장 내 토큰들에 대한 임베딩을 수행한 후 -> pooler를 이용한 문장 임베딩 계산이 일어납니다.
models = SentenceTransformer(modules=[embedding_model, pooler])

# Dataset 로딩

In [9]:
from datasets import load_dataset

datasets = load_dataset("klue", "sts")
datasets

DatasetDict({
    train: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 11668
    })
    validation: Dataset({
        features: ['guid', 'source', 'sentence1', 'sentence2', 'labels'],
        num_rows: 519
    })
})

In [13]:
datasets['train'][0]

{'guid': 'klue-sts-v1_train_00000',
 'source': 'airbnb-rtt',
 'sentence1': '숙소 위치는 찾기 쉽고 일반적인 한국의 반지하 숙소입니다.',
 'sentence2': '숙박시설의 위치는 쉽게 찾을 수 있고 한국의 대표적인 반지하 숙박시설입니다.',
 'labels': {'label': 3.7, 'real-label': 3.714285714285714, 'binary-label': 1}}

# 데이터 전처리
얻어낸 `datasets`를 `SentenceTransformer` 양식에 맞게 바꿔주는 작업을 수행합니다. 이 때 유사도 점수가 0 ~ 5점으로 되어 있는데, 이를 0 ~ 1 사이로 정규화 시켜줍니다.

In [14]:
# InputExample : SentenceTransformer에서 사용되는 데이터 양식으로서 문장과 문장에 대한 결과(label)을 지정할 수 있습니다.
#  데이터를 InputExample(texts=['안녕하세요', '반갑습니다'], label=0.8) 형식으로 관리할 수 있습니다.
from sentence_transformers.readers import InputExample

train_samples = []
validation_samples = []

# KLUE STS 내 훈련, 검증 데이터 예제 변환
for phase in ["train", "validation"]:
  examples = datasets[phase]

  for example in examples:
      score = float(example["labels"]["label"]) / 5.0 # 0.0 ~ 1.0 까지로 스케일링

      inp_example = InputExample(
          texts=[example["sentence1"], example["sentence2"]],
          label=score
      )

      if phase == "validation":
        validation_samples.append(inp_example)
      else:
        train_samples.append(inp_example)

In [16]:
train_samples[0].texts, train_samples[0].label

(['숙소 위치는 찾기 쉽고 일반적인 한국의 반지하 숙소입니다.',
  '숙박시설의 위치는 쉽게 찾을 수 있고 한국의 대표적인 반지하 숙박시설입니다.'],
 0.74)

In [17]:
validation_samples[0].texts, validation_samples[0].label

(['무엇보다도 호스트분들이 너무 친절하셨습니다.', '무엇보다도, 호스트들은 매우 친절했습니다.'], 0.9800000000000001)

# DataLoader 정의

In [19]:
from torch.utils.data import DataLoader

batch_size = 32

train_dataloader = DataLoader(
    train_samples,
    shuffle=True,
    batch_size=batch_size
)

validation_dataloader는 따로 만들지 않고, 대신에 모델의 문장 임베딩 간 코사인 유사도가 얼마나 골드 라벨에 가까운지 계산하는 역할을 수행토록 합니다.

즉 train_dataloader를 이용해 훈련한 결과가 validation_dataloader로 Embedding-Pooling 과정을 진행한 후 두 문장의 유사도를 구한 것을 **평가(Evaluate)**했을 때 얼마나 잘 따라가는지를 본다고 생각하면 쉽습니다.

-------------------------------------------------
이 부분을 좀 더 쉽게 설명하자면, validation_dataloader를 따로 만들지 않고 **평가자(Evaluator)**를 사용해서 검증 과정을 대신하는 방식입니다.

훈련 데이터(train_dataloader)를 사용해 모델을 학습한 후, 검증 단계에서는 두 문장의 유사도를 계산해 그 결과가 **실제 라벨(골드 라벨)**과 얼마나 비슷한지를 평가합니다. 여기서 중요한 점은, 검증(validation) 단계에서는 훈련과 같은 방식으로 데이터를 처리할 필요가 없기 때문에 별도의 validation_dataloader를 만들지 않는다는 것입니다.

대신 EmbeddingSimilarityEvaluator라는 평가자를 사용하여, 미리 지정된 검증 샘플들(validation_samples)의 문장 임베딩을 계산하고, 그 임베딩 벡터들 간의 코사인 유사도를 측정합니다. 이렇게 측정한 유사도가 실제 라벨과 얼마나 가까운지를 통해 모델의 성능을 평가합니다.

따라서, 훈련 데이터로 모델을 학습한 후, 검증 데이터에서 문장 임베딩을 계산한 다음, 두 문장이 얼마나 유사한지(코사인 유사도)를 평가하여 모델이 올바르게 학습되었는지 확인하는 과정으로 이해할 수 있습니다.








---------------------------------------------------------------------
**골드 라벨(Gold Label)**이란, **모델이 학습하거나 평가할 때 사용할 수 있는 "정답 데이터"**를 의미합니다. 즉, 사람이 수작업으로 정확하게 라벨링한 기준 데이터로, 모델이 예측한 결과를 비교하여 평가하는데 사용됩니다.

예시로 설명:
예를 들어, 문장 유사도 평가에서 두 문장의 유사도를 측정할 때, 사람이 직접 평가한 유사도 점수가 골드 라벨입니다. 모델이 두 문장의 유사도를 계산하면, 이 계산된 유사도를 골드 라벨과 비교하여 모델의 성능을 측정합니다.
왜 중요한가?
신뢰성: 골드 라벨은 전문가나 사람이 직접 라벨링한 데이터이므로, 그 정확도가 높다고 간주됩니다.
평가 기준: 모델이 학습하면서 예측한 값들이 얼마나 골드 라벨에 근접하는지에 따라 모델의 성능을 평가할 수 있습니다.
결론적으로, 골드 라벨은 "정답" 역할을 하며, 모델의 예측 결과와 비교해 성능을 평가하는 중요한 기준이 됩니다.

In [20]:
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    validation_samples,
    name="sts-dev"
)

# 모델 훈련

## warmup step
훈련 배치 데이터의 10% 정도를 본격적인 훈련전에 집어 넣어 모델의 학습에 도움이 될 수 있도록 합니다.

본 운동을 하기 전에 준비 운동 같은 단계라고 생각하면 됩니다.

In [25]:
import math

num_epochs = 4
warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1)

## 훈련

In [ ]:
from sentence_transformers import losses
from datetime import datetime

# 두 문장의 유사도를 코사인 유사도를 구한 다음 손실을 구하기 때문에 코사인 유사도 손실 함수를 사용
train_loss = losses.CosineSimilarityLoss(model=models)
model_save_path = "output/training_klue_sts_" + model_name.replace("/", "-") + "-" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

models.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=num_epochs,
    evaluation_steps=1000,
    warmup_steps=warmup_steps,
    output_path=model_save_path
)

Step,Training Loss,Validation Loss,Sts-dev Pearson Cosine,Sts-dev Spearman Cosine,Sts-dev Pearson Manhattan,Sts-dev Spearman Manhattan,Sts-dev Pearson Euclidean,Sts-dev Spearman Euclidean,Sts-dev Pearson Dot,Sts-dev Spearman Dot,Sts-dev Pearson Max,Sts-dev Spearman Max
365,No log,No log,0.875468,0.869097,0.873406,0.866239,0.874419,0.866693,0.861046,0.851442,0.875468,0.869097
730,0.027200,No log,0.882968,0.881957,0.884404,0.880098,0.885244,0.880877,0.872224,0.867258,0.885244,0.881957
1000,0.005000,No log,0.886587,0.885416,0.888436,0.884114,0.889325,0.885150,0.874892,0.869687,0.889325,0.885416
1095,0.005000,No log,0.888420,0.888415,0.890116,0.885687,0.890642,0.886206,0.877493,0.873330,0.890642,0.888415
1460,0.005000,No log,0.888688,0.889012,0.890156,0.886228,0.890923,0.886839,0.877576,0.873944,0.890923,0.889012


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

# 테스트 세트로 평가

In [ ]:
# 테스트를 위해 다른 종류의 STS 데이터 세트인 KorSTS 로딩 및 SentenceTransformer에 맞게 처리
testsets = load_dataset("kor_nlu", "sts")

testsets

The repository for kor_nlu contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/kor_nlu.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
The repository for kor_nlu contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/kor_nlu.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/5703 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1471 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['genre', 'filename', 'year', 'id', 'score', 'sentence1', 'sentence2'],
        num_rows: 5703
    })
    validation: Dataset({
        features: ['genre', 'filename', 'year', 'id', 'score', 'sentence1', 'sentence2'],
        num_rows: 1471
    })
    test: Dataset({
        features: ['genre', 'filename', 'year', 'id', 'score', 'sentence1', 'sentence2'],
        num_rows: 1379
    })
})

In [ ]:
test_samples = []

# KorSTS 내 테스트 데이터 예제 변환
for example in testsets["test"]:
    score = float(example["score"]) / 5.0

    if example["sentence1"] and example["sentence2"]:
        inp_example = InputExample(
            texts=[example["sentence1"], example["sentence2"]],
            label=score,
        )

    test_samples.append(inp_example)

In [ ]:
loadded_model = SentenceTransformer(model_save_path)
test_evaluator = EmbeddingSimilarityEvaluator.from_input_examples(test_samples)

In [ ]:
test_evaluator(loadded_model, output_path=model_save_path)

{'pearson_cosine': 0.7702232404973869,
 'spearman_cosine': 0.7608371989442981,
 'pearson_manhattan': 0.7659247360429031,
 'spearman_manhattan': 0.764656945939528,
 'pearson_euclidean': 0.7658076542919648,
 'spearman_euclidean': 0.764476468092129,
 'pearson_dot': 0.7417271657447979,
 'spearman_dot': 0.7304753514347492,
 'pearson_max': 0.7702232404973869,
 'spearman_max': 0.764656945939528}

데이터가 만들어진 과정이 KLUE의 STS와 KorSTS가 다르기 때문에 성능이 매우 뛰어나진 않아보기에 테스트 된 것을 알 수 있습니다.

# SentenceTransformer 활용

## 시멘틱 서치
입력된 문장 간 유사도를 쉽고 빠르게 구할 수 있도록 설계된 `sentence-transformers`를 이용한다면 임베딩을 활용해 다양한 어플리케이션을 고안할 수 있습니다.

먼저 여러 문장 후보군이 주어졌을 때, 입력된 문장과 가장 유사한 문장을 계산하는 예제를 살펴보도록 합시다.

이를 위해 검색의 대상이 되는 문장 후보군을 다음과 같이 정의할 필요가 있습니다. 이후, 정의된 문장 후보군을 미리 임베딩합니다.

In [ ]:
docs = [
    "1992년 7월 8일 손흥민은 강원도 춘천시 후평동에서 아버지 손웅정과 어머니 길은자의 차남으로 태어나 그곳에서 자랐다.",
    "형은 손흥윤이다.",
    "춘천 부안초등학교를 졸업했고, 춘천 후평중학교에 입학한 후 2학년때 원주 육민관중학교 축구부에 들어가기 위해 전학하여 졸업하였으며, 2008년 당시 FC 서울의 U-18팀이었던 동북고등학교 축구부에서 선수 활동 중 대한축구협회 우수선수 해외유학 프로젝트에 선발되어 2008년 8월 독일 분데스리가의 함부르크 유소년팀에 입단하였다.",
    "함부르크 유스팀 주전 공격수로 2008년 6월 네덜란드에서 열린 4개국 경기에서 4게임에 출전, 3골을 터뜨렸다.",
    "1년간의 유학 후 2009년 8월 한국으로 돌아온 후 10월에 개막한 FIFA U-17 월드컵에 출전하여 3골을 터트리며 한국을 8강으로 이끌었다.",
    "그해 11월 함부르크의 정식 유소년팀 선수 계약을 체결하였으며 독일 U-19 리그 4경기 2골을 넣고 2군 리그에 출전을 시작했다.",
    "독일 U-19 리그에서 손흥민은 11경기 6골, 2부 리그에서는 6경기 1골을 넣으며 재능을 인정받아 2010년 6월 17세의 나이로 함부르크의 1군 팀 훈련에 참가, 프리시즌 활약으로 함부르크와 정식 계약을 한 후 10월 18세에 함부르크 1군 소속으로 독일 분데스리가에 데뷔하였다.",
]

document_embeddings = loadded_model.encode(docs)
document_embeddings

array([[-0.27014086,  0.38607192,  0.29859635, ...,  0.05982703,
        -0.49910095,  0.06126191],
       [-0.03964891,  0.08223528,  0.14235209, ..., -0.51654285,
        -0.51922846, -0.16850345],
       [-0.12232523,  0.05636293, -0.250255  , ...,  0.38720357,
        -0.82892257,  0.2607616 ],
       ...,
       [-0.6124201 , -0.2016571 ,  0.22940885, ..., -0.11177877,
        -0.6543939 ,  0.78529245],
       [ 0.04260467,  0.38919744, -0.35943198, ...,  0.13646524,
        -1.1172526 ,  0.01331137],
       [ 0.06677505,  0.25976148, -0.16866627, ...,  0.25849375,
        -0.9848711 , -0.37901172]], dtype=float32)

In [ ]:
# 테스트 할 문장
query = "손흥민은 어린 나이에 유럽에 진출하였다."

query_embedding = loadded_model.encode(query)
print(query_embedding)

[ 2.51015425e-01  7.18877539e-02  3.18499535e-01  1.75983876e-01
  3.71476591e-01 -1.03105567e-01  3.01482342e-02 -2.05017060e-01
 -3.66159707e-01 -1.72114953e-01 -4.07421529e-01 -7.78742060e-02
 -1.14076555e-01 -4.03025776e-01  4.45655555e-01  1.14237773e+00
 -5.50697327e-01 -4.00348037e-01  2.02992409e-01  1.60568818e-01
 -3.21738303e-01 -6.69185519e-01 -4.75066938e-02  1.13031436e-02
 -2.77938664e-01 -1.41672820e-01 -6.42343387e-02  7.17299506e-02
  3.19587857e-01 -6.58050120e-01  9.14276391e-02  2.23660842e-01
 -1.07488923e-01 -4.13736552e-01  3.92496288e-01  1.81215733e-01
 -2.53290385e-01  1.62742123e-01  2.64080435e-01 -1.50213450e-01
 -2.15333570e-02 -5.72196126e-01  6.36787295e-01 -2.40618259e-01
  1.41320646e-01  4.26984578e-01  7.14188665e-02 -1.66867733e-01
 -3.69097024e-01  3.40135187e-01  5.48947930e-01  3.99114564e-02
  1.64143801e-01  3.82650904e-02 -5.76999366e-01 -9.93260369e-02
 -1.31702080e-01  6.49326980e-01 -2.45115727e-01  2.50349253e-01
  4.44177181e-01  3.10436

In [ ]:
from sentence_transformers import util
import torch

top_k = 5

# 입력 문장 - 문장 후보군 간 코사인 유사도 계산하기
cos_scores = util.pytorch_cos_sim(
    query_embedding,
    document_embeddings
)[0]

# 코사인 유사도 순으로 top_k개 문장 추출
top_results = torch.topk(cos_scores, k=top_k)

In [ ]:
top_results

torch.return_types.topk(
values=tensor([0.5706, 0.4493, 0.3729, 0.3697, 0.3050]),
indices=tensor([6, 5, 3, 0, 2]))

In [ ]:
print(f"입력 문장: {query}")
print(f"\n<입력 문장과 유사한 {top_k} 개의 문장>\n")

for i, (score, idx) in enumerate(zip(top_results[0], top_results[1])):
    print(f"{i+1}: {docs[idx]} {'(유사도: {:.4f})'.format(score)}\n")

입력 문장: 손흥민은 어린 나이에 유럽에 진출하였다.

<입력 문장과 유사한 5 개의 문장>

1: 독일 U-19 리그에서 손흥민은 11경기 6골, 2부 리그에서는 6경기 1골을 넣으며 재능을 인정받아 2010년 6월 17세의 나이로 함부르크의 1군 팀 훈련에 참가, 프리시즌 활약으로 함부르크와 정식 계약을 한 후 10월 18세에 함부르크 1군 소속으로 독일 분데스리가에 데뷔하였다. (유사도: 0.5706)

2: 그해 11월 함부르크의 정식 유소년팀 선수 계약을 체결하였으며 독일 U-19 리그 4경기 2골을 넣고 2군 리그에 출전을 시작했다. (유사도: 0.4493)

3: 함부르크 유스팀 주전 공격수로 2008년 6월 네덜란드에서 열린 4개국 경기에서 4게임에 출전, 3골을 터뜨렸다. (유사도: 0.3729)

4: 1992년 7월 8일 손흥민은 강원도 춘천시 후평동에서 아버지 손웅정과 어머니 길은자의 차남으로 태어나 그곳에서 자랐다. (유사도: 0.3697)

5: 춘천 부안초등학교를 졸업했고, 춘천 후평중학교에 입학한 후 2학년때 원주 육민관중학교 축구부에 들어가기 위해 전학하여 졸업하였으며, 2008년 당시 FC 서울의 U-18팀이었던 동북고등학교 축구부에서 선수 활동 중 대한축구협회 우수선수 해외유학 프로젝트에 선발되어 2008년 8월 독일 분데스리가의 함부르크 유소년팀에 입단하였다. (유사도: 0.3050)



# 클러스터링

In [ ]:
from sklearn.cluster import KMeans

document_embeddings = loadded_model.encode(docs)

num_clusters = 3

k_means = KMeans(n_clusters=num_clusters)
k_means.fit(document_embeddings)

KMeans(n_clusters=3)

In [ ]:
cluster_assignment = k_means.labels_

In [ ]:
cluster_assignment

array([2, 2, 0, 1, 0, 1, 1], dtype=int32)

In [ ]:
# 클러스터 개수 만큼 문장을 담을 리스트 초기화
clustered_sentences = [[] for _ in range(num_clusters)]

# 클러스터링 결과를 돌며 각 클러스터에 맞게 문장 삽입
for sentence_id, cluster_id in enumerate(cluster_assignment):
    clustered_sentences[cluster_id].append(docs[sentence_id])

for i, cluster in enumerate(clustered_sentences):
    result = "\n".join(cluster)
    print(f"< 클러스터 {i+1} >\n{result}\n")

< 클러스터 1 >
춘천 부안초등학교를 졸업했고, 춘천 후평중학교에 입학한 후 2학년때 원주 육민관중학교 축구부에 들어가기 위해 전학하여 졸업하였으며, 2008년 당시 FC 서울의 U-18팀이었던 동북고등학교 축구부에서 선수 활동 중 대한축구협회 우수선수 해외유학 프로젝트에 선발되어 2008년 8월 독일 분데스리가의 함부르크 유소년팀에 입단하였다.
1년간의 유학 후 2009년 8월 한국으로 돌아온 후 10월에 개막한 FIFA U-17 월드컵에 출전하여 3골을 터트리며 한국을 8강으로 이끌었다.

< 클러스터 2 >
함부르크 유스팀 주전 공격수로 2008년 6월 네덜란드에서 열린 4개국 경기에서 4게임에 출전, 3골을 터뜨렸다.
그해 11월 함부르크의 정식 유소년팀 선수 계약을 체결하였으며 독일 U-19 리그 4경기 2골을 넣고 2군 리그에 출전을 시작했다.
독일 U-19 리그에서 손흥민은 11경기 6골, 2부 리그에서는 6경기 1골을 넣으며 재능을 인정받아 2010년 6월 17세의 나이로 함부르크의 1군 팀 훈련에 참가, 프리시즌 활약으로 함부르크와 정식 계약을 한 후 10월 18세에 함부르크 1군 소속으로 독일 분데스리가에 데뷔하였다.

< 클러스터 3 >
1992년 7월 8일 손흥민은 강원도 춘천시 후평동에서 아버지 손웅정과 어머니 길은자의 차남으로 태어나 그곳에서 자랐다.
형은 손흥윤이다.



In [ ]:
# 클러스터 개수 만큼 문장을 담을 리스트 초기화
clustered_sentences = [[] for _ in range(num_clusters)]

# 클러스터링 결과를 돌며 각 클러스터에 맞게 문장 삽입
for sentence_id, cluster_id in enumerate(cluster_assignment):
    clustered_sentences[cluster_id].append(docs[sentence_id])

for i, cluster in enumerate(clustered_sentences):
    result = "\n".join(cluster)
    print(f"< 클러스터 {i+1} >\n{result}\n")

< 클러스터 1 >
춘천 부안초등학교를 졸업했고, 춘천 후평중학교에 입학한 후 2학년때 원주 육민관중학교 축구부에 들어가기 위해 전학하여 졸업하였으며, 2008년 당시 FC 서울의 U-18팀이었던 동북고등학교 축구부에서 선수 활동 중 대한축구협회 우수선수 해외유학 프로젝트에 선발되어 2008년 8월 독일 분데스리가의 함부르크 유소년팀에 입단하였다.
1년간의 유학 후 2009년 8월 한국으로 돌아온 후 10월에 개막한 FIFA U-17 월드컵에 출전하여 3골을 터트리며 한국을 8강으로 이끌었다.

< 클러스터 2 >
함부르크 유스팀 주전 공격수로 2008년 6월 네덜란드에서 열린 4개국 경기에서 4게임에 출전, 3골을 터뜨렸다.
그해 11월 함부르크의 정식 유소년팀 선수 계약을 체결하였으며 독일 U-19 리그 4경기 2골을 넣고 2군 리그에 출전을 시작했다.
독일 U-19 리그에서 손흥민은 11경기 6골, 2부 리그에서는 6경기 1골을 넣으며 재능을 인정받아 2010년 6월 17세의 나이로 함부르크의 1군 팀 훈련에 참가, 프리시즌 활약으로 함부르크와 정식 계약을 한 후 10월 18세에 함부르크 1군 소속으로 독일 분데스리가에 데뷔하였다.

< 클러스터 3 >
1992년 7월 8일 손흥민은 강원도 춘천시 후평동에서 아버지 손웅정과 어머니 길은자의 차남으로 태어나 그곳에서 자랐다.
형은 손흥윤이다.



기존에 있던 모델 사용

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("Huffon/sentence-klue-roberta-base")

docs = [
    "1992년 7월 8일 손흥민은 강원도 춘천시 후평동에서 아버지 손웅정과 어머니 길은자의 차남으로 태어나 그곳에서 자랐다.",
    "형은 손흥윤이다.",
    "춘천 부안초등학교를 졸업했고, 춘천 후평중학교에 입학한 후 2학년때 원주 육민관중학교 축구부에 들어가기 위해 전학하여 졸업하였으며, 2008년 당시 FC 서울의 U-18팀이었던 동북고등학교 축구부에서 선수 활동 중 대한축구협회 우수선수 해외유학 프로젝트에 선발되어 2008년 8월 독일 분데스리가의 함부르크 유소년팀에 입단하였다.",
    "함부르크 유스팀 주전 공격수로 2008년 6월 네덜란드에서 열린 4개국 경기에서 4게임에 출전, 3골을 터뜨렸다.",
    "1년간의 유학 후 2009년 8월 한국으로 돌아온 후 10월에 개막한 FIFA U-17 월드컵에 출전하여 3골을 터트리며 한국을 8강으로 이끌었다.",
    "그해 11월 함부르크의 정식 유소년팀 선수 계약을 체결하였으며 독일 U-19 리그 4경기 2골을 넣고 2군 리그에 출전을 시작했다.",
    "독일 U-19 리그에서 손흥민은 11경기 6골, 2부 리그에서는 6경기 1골을 넣으며 재능을 인정받아 2010년 6월 17세의 나이로 함부르크의 1군 팀 훈련에 참가, 프리시즌 활약으로 함부르크와 정식 계약을 한 후 10월 18세에 함부르크 1군 소속으로 독일 분데스리가에 데뷔하였다.",
]
document_embeddings = model.encode(docs)

query = "손흥민은 어린 나이에 유럽에 진출하였다."
query_embedding = model.encode(query)

top_k = min(5, len(docs))

# 입력 문장 - 문장 후보군 간 코사인 유사도 계산 후,
cos_scores = util.pytorch_cos_sim(query_embedding, document_embeddings)[0]

# 코사인 유사도 순으로 `top_k` 개 문장 추출
top_results = torch.topk(cos_scores, k=top_k)

print(f"입력 문장: {query}")
print(f"\n<입력 문장과 유사한 {top_k} 개의 문장>\n")

for i, (score, idx) in enumerate(zip(top_results[0], top_results[1])):
    print(f"{i+1}: {docs[idx]} {'(유사도: {:.4f})'.format(score)}\n")

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


입력 문장: 손흥민은 어린 나이에 유럽에 진출하였다.

<입력 문장과 유사한 5 개의 문장>

1: 독일 U-19 리그에서 손흥민은 11경기 6골, 2부 리그에서는 6경기 1골을 넣으며 재능을 인정받아 2010년 6월 17세의 나이로 함부르크의 1군 팀 훈련에 참가, 프리시즌 활약으로 함부르크와 정식 계약을 한 후 10월 18세에 함부르크 1군 소속으로 독일 분데스리가에 데뷔하였다. (유사도: 0.5897)

2: 그해 11월 함부르크의 정식 유소년팀 선수 계약을 체결하였으며 독일 U-19 리그 4경기 2골을 넣고 2군 리그에 출전을 시작했다. (유사도: 0.4857)

3: 1992년 7월 8일 손흥민은 강원도 춘천시 후평동에서 아버지 손웅정과 어머니 길은자의 차남으로 태어나 그곳에서 자랐다. (유사도: 0.4047)

4: 함부르크 유스팀 주전 공격수로 2008년 6월 네덜란드에서 열린 4개국 경기에서 4게임에 출전, 3골을 터뜨렸다. (유사도: 0.3953)

5: 춘천 부안초등학교를 졸업했고, 춘천 후평중학교에 입학한 후 2학년때 원주 육민관중학교 축구부에 들어가기 위해 전학하여 졸업하였으며, 2008년 당시 FC 서울의 U-18팀이었던 동북고등학교 축구부에서 선수 활동 중 대한축구협회 우수선수 해외유학 프로젝트에 선발되어 2008년 8월 독일 분데스리가의 함부르크 유소년팀에 입단하였다. (유사도: 0.3183)

